# KOSPI 단일 정책 PPO 재실험 — 백본 민감도 + 누수·버그 수정 (2026-10-03)

**목적**: 논문 1.1절의 "KOSPI 단일 정책 PPO 실패(+0.28% vs 등가중 +34.33%)"가 백본·버그와 무관한
결과인지 확인한다. 같은 PPO 코드·시드로 Chronos+iTransformer 임베딩(05-16)은 **+137.22%**,
Chronos+MTGNN 임베딩(05-22)은 **+0.28%**가 나왔다. 그런데 원래 파이프라인에는 아래 네 가지 문제가 있었다.

| # | 문제 | 이 노트북의 수정 |
|---|---|---|
| 1 | 백본 학습 분할이 `TRAIN_END=2024-06-30`, `VAL_END=2024-12-31` → RL 테스트(2024–2025)와 겹침 | 백본은 **목표 수익률 날짜가 2023-12-31 이전**인 샘플로만 학습·검증 |
| 2 | RL 임베딩 추출 시 피처 순서가 학습 때와 다름(학습: `SMA_5`,`Return_5d` / 추출: `BB_upper`,`BB_lower`) | 학습·추출 모두 `FEATURE_COLS` 하나만 사용 |
| 3 | RL 임베딩 추출 시 RevIN 정규화 미적용(원시 가격 스케일 입력) | 학습·추출 모두 같은 `revin()` 함수 사용 |
| 4 | DTW 클러스터링 기간 2022–2024 → 테스트 구간 정보 누수 | **2021–2023**으로만 재클러스터링 |

**실험 설계**: 백본 2종(Chronos+iTransformer, Chronos+MTGNN) × PPO 시드 5개. PPO·환경·하이퍼파라미터는
`rl_trading_colab.ipynb`와 동일하다(학습 2021–2023, 테스트 2024–2025, 거래비용 0.1%, 300k 스텝).

**예상 소요 (T4 GPU)**: 클러스터링 ~10분 · 백본 학습 2종 ~25분 · RL 임베딩 추출 ~30분 · PPO 10회 × ~25분.
PPO 결과는 실행 1회마다 Drive에 저장되므로, 런타임이 끊기면 §0~§2를 다시 실행한 뒤 §6부터 이어서 돌리면
이미 끝난 (백본, 시드)는 건너뛴다.

> 런타임 → 런타임 유형 변경 → **T4 GPU**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q chronos-forecasting stable-baselines3[extra] gymnasium tslearn h5py pyarrow

## 0. 설정

In [ ]:
import gc, json, warnings
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from pathlib import Path
from scipy.stats import spearmanr
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

DRIVE_ROOT = Path('/content/drive/MyDrive/졸업프로젝트')
DATA_DIR   = DRIVE_ROOT / 'data'
PARQUET    = DATA_DIR / 'kospi_valid.parquet'
OLD_CLUSTER_CSV = DATA_DIR / 'cluster_assignments.csv'
OUT_DIR    = DATA_DIR / 'kospi_rerun'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# sequences.h5(백본 학습 데이터)와 동일한 20개 피처·순서 — 학습과 추출 모두 이 리스트만 사용
FEATURE_COLS = ['Adj_Close', 'Open', 'High', 'Low', 'Volume',
                'SMA_5', 'SMA_20', 'SMA_60', 'EMA_12', 'EMA_26',
                'MACD', 'MACD_signal', 'MACD_hist', 'RSI_14', 'BB_width',
                'Volume_ratio', 'Return_1d', 'Return_5d', 'Volatility_20d', 'ATR_14']
N_FEATURES = len(FEATURE_COLS)
SEQ_LEN, PRED_LEN = 60, 5

# 백본: 목표 수익률(5일 뒤) 날짜 기준으로 분할 → RL 테스트(2024~)와 겹치지 않음
BB_TRAIN_END = '2023-06-30'
BB_VAL_END   = '2023-12-31'
MAX_TRAIN, MAX_VAL = 80_000, 30_000

# 클러스터링: RL 학습 구간까지만
CLUSTER_START, CLUSTER_END = '2021-01-01', '2023-12-31'
N_CLUSTERS, MIN_DAYS = 5, 700

# RL: rl_trading_colab.ipynb와 동일
RL_TRAIN_START, RL_TRAIN_END = '2021-01-01', '2023-12-31'
RL_TEST_START,  RL_TEST_END  = '2024-01-01', '2025-12-31'
TC = 0.001
PPO_TIMESTEPS = 300_000

BACKBONES = ['itransformer', 'mtgnn']
SEEDS = [42, 11, 22, 33, 44]

torch.manual_seed(42); np.random.seed(42)
print('설정 완료 →', OUT_DIR)

In [ ]:
raw = pd.read_parquet(PARQUET)
raw['Date'] = pd.to_datetime(raw['Date'])
raw['종목코드'] = raw['종목코드'].astype(str).str.zfill(6)
raw = raw.sort_values(['종목코드', 'Date']).reset_index(drop=True)
missing = [c for c in FEATURE_COLS + ['RetTarget_1d', 'Return_1d'] if c not in raw.columns]
assert not missing, f'parquet에 없는 컬럼: {missing}'
print(raw.shape, raw['Date'].min().date(), '~', raw['Date'].max().date(), '|', raw['종목코드'].nunique(), '종목')

# 종목별 배열로 한 번만 변환해 둔다
tickers = raw['종목코드'].unique().tolist()
per_ticker = {}
for tk, g in raw.groupby('종목코드', sort=False):
    per_ticker[tk] = {
        'feat': g[FEATURE_COLS].to_numpy(np.float32),
        'close': g['Adj_Close'].to_numpy(np.float64),
        'dates': g['Date'].dt.strftime('%Y-%m-%d').to_numpy(),
    }
print('종목 배열 준비 완료:', len(per_ticker))

def revin(w):
    # 윈도우별 정규화 — preprocess_sequences의 RevIN과 동일(nanmean/nanstd + 1e-8)
    mu = np.nanmean(w, axis=0, keepdims=True)
    sd = np.nanstd(w, axis=0, keepdims=True) + 1e-8
    return ((w - mu) / sd).astype(np.float32)

## 1. DTW 재클러스터링 (2021–2023만 사용)

원래 `dtw_clustering_colab.ipynb`와 같은 방법(월말 누적수익률, 종목별 z-score, soft-DTW K-means)이며, 기간만 2021–2023으로 바꾸고 K는 원래와 같은 5로 고정한다.

In [ ]:
%%time
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.preprocessing import StandardScaler
from tslearn.clustering import TimeSeriesKMeans
from tslearn.utils import to_time_series_dataset

period = raw[(raw['Date'] >= CLUSTER_START) & (raw['Date'] <= CLUSTER_END)]
valid_codes = period.groupby('종목코드').size().loc[lambda s: s >= MIN_DAYS].index
period = period[period['종목코드'].isin(valid_codes)]
ret_pivot = period.pivot_table(index='Date', columns='종목코드', values='Return_1d').sort_index().fillna(0)
cum_monthly = ((1 + ret_pivot).cumprod() - 1).resample('ME').last()
series_norm = StandardScaler().fit_transform(cum_monthly.values).T   # (N, T_month)
codes_list = list(ret_pivot.columns)

km = TimeSeriesKMeans(n_clusters=N_CLUSTERS, metric='softdtw', metric_params={'gamma': 0.1},
                      max_iter=50, tol=1e-4, random_state=42, n_jobs=-1)
labels = km.fit_predict(to_time_series_dataset(series_norm))
sil = silhouette_score(series_norm, labels, metric='euclidean')

new_clusters = pd.DataFrame({'종목코드': codes_list, 'cluster': labels})
new_clusters.to_csv(OUT_DIR / 'cluster_assignments_2021_2023.csv', index=False, encoding='utf-8-sig')
print(f'대상 {len(codes_list)}종목 | 실루엣 {sil:.4f}')
print(new_clusters['cluster'].value_counts().sort_index().to_dict())

old = pd.read_csv(OLD_CLUSTER_CSV, dtype={'종목코드': str})
old['종목코드'] = old['종목코드'].str.zfill(6)
m = new_clusters.merge(old[['종목코드', 'cluster']], on='종목코드', suffixes=('_new', '_old'))
print(f'기존(2022–2024) 클러스터와의 ARI: {adjusted_rand_score(m.cluster_old, m.cluster_new):.3f} (공통 {len(m)}종목)')

## 2. 백본 학습 데이터 구성

윈도우 `[e-59, e]`(e일 포함)를 입력으로, `e`일 종가 대비 `e+5`일 종가 수익률을 타겟으로 한다. **타겟 날짜(e+5)** 가 `BB_TRAIN_END` 이하면 학습, `BB_VAL_END` 이하면 검증에 넣는다.

In [ ]:
%%time
rng = np.random.default_rng(42)
cand = {'train': [], 'val': []}
for tk, d in per_ticker.items():
    feat, close, dates = d['feat'], d['close'], d['dates']
    bad = np.isnan(feat).any(axis=1) | (close <= 0)
    bad_cum = np.concatenate([[0], np.cumsum(bad)])
    for e in range(SEQ_LEN - 1, len(dates) - PRED_LEN):
        if bad_cum[e + 1] - bad_cum[e - SEQ_LEN + 1] > 0 or close[e + PRED_LEN] <= 0:
            continue
        tgt_date = dates[e + PRED_LEN]
        if tgt_date <= BB_TRAIN_END:
            cand['train'].append((tk, e))
        elif dates[e] > BB_TRAIN_END and tgt_date <= BB_VAL_END:
            cand['val'].append((tk, e))

def build(pairs, cap):
    if len(pairs) > cap:
        pairs = [pairs[i] for i in np.sort(rng.choice(len(pairs), cap, replace=False))]
    X = np.empty((len(pairs), SEQ_LEN, N_FEATURES), np.float32)
    y = np.empty(len(pairs), np.float32)
    for i, (tk, e) in enumerate(pairs):
        d = per_ticker[tk]
        X[i] = revin(d['feat'][e - SEQ_LEN + 1:e + 1])
        y[i] = d['close'][e + PRED_LEN] / d['close'][e] - 1
    return X, y

X_tr, y_tr = build(cand['train'], MAX_TRAIN)
X_va, y_va = build(cand['val'], MAX_VAL)
for n, y in [('train', y_tr), ('val', y_va)]:
    print(f'{n}: {len(y):,}개 | 상승비율 {(y > 0).mean():.3f}')

## 3. Chronos 인코더 임베딩

원래 학습과 같이 **RevIN된 종가(피처 0번)** 를 Chronos에 넣는다. RL 추출(§5)도 같은 입력을 쓴다.

In [ ]:
from chronos import ChronosPipeline
pipeline = ChronosPipeline.from_pretrained('amazon/chronos-t5-small', device_map=DEVICE, torch_dtype=torch.float32)

@torch.no_grad()
def chronos_embs(close_np, batch_size=512):
    tok, enc, out = pipeline.tokenizer, pipeline.model.model.encoder, []
    for i in range(0, len(close_np), batch_size):
        ctx = torch.tensor(close_np[i:i + batch_size], dtype=torch.float32)   # tokenizer는 CPU 텐서 필요
        ids, mask, _ = tok.context_input_transform(ctx)
        h = enc(input_ids=ids.to(DEVICE), attention_mask=mask.to(DEVICE)).last_hidden_state
        m = mask.to(DEVICE).unsqueeze(-1).float()
        out.append(((h * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy())
    return np.concatenate(out)

C_tr = chronos_embs(X_tr[:, :, 0]); C_va = chronos_embs(X_va[:, :, 0])
print(C_tr.shape, C_va.shape)

## 4. 하이브리드 백본 2종 학습

모델 정의·하이퍼파라미터는 원래 노트북과 같다(iTransformer: `d_model=128, nhead=4, n_layers=2` / MTGNN: `d_model=64, n_layers=3`). 검증 MAE로 early stopping.

In [ ]:
class ChronosITransformer(nn.Module):
    # 05-16 hybrid_model_colab.ipynb(커밋 279587b)의 정의 그대로
    def __init__(self, seq_len=60, n_features=20, chronos_dim=512, d_model=128, nhead=4, n_layers=2, dropout=0.1):
        super().__init__()
        self.chronos_proj = nn.Linear(chronos_dim, d_model)
        self.chronos_norm = nn.LayerNorm(d_model)
        self.variate_embed = nn.Linear(seq_len, d_model)
        layer = nn.TransformerEncoderLayer(d_model, nhead, d_model * 4, dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers)
        self.it_norm = nn.LayerNorm(d_model)
        self.fusion = nn.Sequential(nn.Linear(d_model + n_features * d_model, 256), nn.GELU(), nn.Dropout(dropout),
                                    nn.Linear(256, 64), nn.GELU())
        self.head = nn.Linear(64, 1)
    def _fuse(self, x, c):
        it = self.it_norm(self.transformer(self.variate_embed(x.permute(0, 2, 1)))).flatten(1)
        return self.fusion(torch.cat([self.chronos_norm(self.chronos_proj(c)), it], -1))
    def forward(self, x, c): return self.head(self._fuse(x, c)).squeeze(-1)
    @torch.no_grad()
    def embed(self, x, c): self.eval(); return self._fuse(x, c)

class AdaptiveGraphLearner(nn.Module):
    def __init__(self, n_nodes, emb_dim=10):
        super().__init__()
        self.E1 = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.1)
        self.E2 = nn.Parameter(torch.randn(n_nodes, emb_dim) * 0.1)
    def forward(self): return torch.softmax(torch.relu(self.E1 @ self.E2.T), dim=-1)

class MTGNNEncoder(nn.Module):
    # hybrid_model_colab.ipynb / crypto/models/hybrid_model.py와 동일
    def __init__(self, seq_len=60, n_features=20, d_model=64, n_layers=3, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Conv2d(1, d_model, (1, 1))
        dil = [1, 2, 4][:n_layers]
        self.filter_convs = nn.ModuleList([nn.Conv2d(d_model, d_model, (1, 3), dilation=(1, d), padding=(0, d)) for d in dil])
        self.gate_convs = nn.ModuleList([nn.Conv2d(d_model, d_model, (1, 3), dilation=(1, d), padding=(0, d)) for d in dil])
        self.res_convs = nn.ModuleList([nn.Conv2d(d_model, d_model, 1) for _ in dil])
        self.gc_weights = nn.ModuleList([nn.Linear(d_model, d_model, bias=False) for _ in dil])
        self.norms = nn.ModuleList([nn.LayerNorm(d_model) for _ in dil])
        self.graph = AdaptiveGraphLearner(n_features)
        self.drop = nn.Dropout(dropout)
    def forward(self, x_flat):
        x = self.input_proj(x_flat.permute(0, 2, 1).unsqueeze(1))
        A = self.graph()
        for fc, gc, rc, gw, norm in zip(self.filter_convs, self.gate_convs, self.res_convs, self.gc_weights, self.norms):
            res, T = x, x.size(-1)
            h = torch.tanh(fc(x)[..., :T]) * torch.sigmoid(gc(x)[..., :T])
            h = gw(torch.einsum('nm,btmc->btnc', A, h.permute(0, 3, 2, 1))).permute(0, 3, 2, 1)
            x = norm((self.drop(h) + rc(res)).permute(0, 3, 2, 1)).permute(0, 3, 2, 1)
        return x.mean(-1).permute(0, 2, 1).flatten(1)

class ChronosMTGNN(nn.Module):
    def __init__(self, seq_len=60, n_features=20, chronos_dim=512, d_model=64, n_layers=3, dropout=0.1):
        super().__init__()
        self.chronos_proj = nn.Linear(chronos_dim, d_model)
        self.chronos_norm = nn.LayerNorm(d_model)
        self.mtgnn = MTGNNEncoder(seq_len, n_features, d_model, n_layers, dropout)
        self.fusion = nn.Sequential(nn.Linear(d_model + n_features * d_model, 256), nn.GELU(), nn.Dropout(dropout),
                                    nn.Linear(256, 64), nn.GELU())
        self.head = nn.Linear(64, 1)
    def _fuse(self, x, c): return self.fusion(torch.cat([self.chronos_norm(self.chronos_proj(c)), self.mtgnn(x)], -1))
    def forward(self, x, c): return self.head(self._fuse(x, c)).squeeze(-1)
    @torch.no_grad()
    def embed(self, x, c): self.eval(); return self._fuse(x, c)

def make_model(name):
    return ChronosITransformer() if name == 'itransformer' else ChronosMTGNN()

In [ ]:
%%time
from torch.utils.data import TensorDataset, DataLoader

def metrics(y, p):
    return {'dir_acc': float(((y > 0) == (p > 0)).mean()), 'spearman': float(spearmanr(y, p)[0]),
            'mae': float(np.abs(y - p).mean())}

def train_backbone(name, epochs=15, patience=3, lr=5e-4, bs=256):
    torch.manual_seed(42)
    model = make_model(name).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    tr = DataLoader(TensorDataset(torch.from_numpy(X_tr), torch.from_numpy(C_tr), torch.from_numpy(y_tr)), bs, shuffle=True)
    Xv, Cv = torch.from_numpy(X_va).to(DEVICE), torch.from_numpy(C_va).to(DEVICE)
    best, bad, state = 1e9, 0, None
    for ep in range(1, epochs + 1):
        model.train()
        for x, c, y in tr:
            opt.zero_grad()
            loss = nn.functional.mse_loss(model(x.to(DEVICE), c.to(DEVICE)), y.to(DEVICE))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        sch.step(); model.eval()
        with torch.no_grad():
            pv = torch.cat([model(Xv[i:i + 2048], Cv[i:i + 2048]) for i in range(0, len(Xv), 2048)]).cpu().numpy()
        m = metrics(y_va, pv)
        print(f'[{name}] ep{ep:02d} val {m}')
        if m['mae'] < best:
            best, bad, best_m = m['mae'], 0, m
            state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience: print('  early stop'); break
    model.load_state_dict(state); model.eval()
    return model, best_m

models, bb_metrics = {}, {}
for name in BACKBONES:
    models[name], bb_metrics[name] = train_backbone(name)
    torch.save(models[name].state_dict(), OUT_DIR / f'hybrid_{name}_2023.pt')
majority = max((y_va > 0).mean(), (y_va <= 0).mean())
print('\n검증 구간 다수 클래스 기준선 dir_acc:', round(float(majority), 4))
json.dump({'val_metrics': bb_metrics, 'val_majority_dir_acc': float(majority)},
          open(OUT_DIR / 'backbone_val_metrics.json', 'w'), ensure_ascii=False, indent=2)

## 5. RL 임베딩 추출 (두 백본 동시)

원래 규칙대로 날짜 `t`의 임베딩은 윈도우 `[t-60, t-1]`(t일 미포함)로 만든다. 학습과 같은 `FEATURE_COLS`·`revin()`을 쓴다. Chronos 임베딩은 두 백본이 공유하므로 한 번만 계산한다.

In [ ]:
%%time
recs = {n: {'tk': [], 'dt': [], 'emb': []} for n in BACKBONES}
for tk in tqdm(tickers, desc='종목'):
    d = per_ticker[tk]
    feat, close, dates = d['feat'], d['close'], d['dates']
    W, D = [], []
    for t in range(SEQ_LEN, len(dates)):
        w = feat[t - SEQ_LEN:t]
        if np.isnan(w).any() or (close[t - SEQ_LEN:t] <= 0).any():
            continue
        W.append(revin(w)); D.append(dates[t])
    if not W:
        continue
    W = np.stack(W)
    C = chronos_embs(W[:, :, 0])
    for n in BACKBONES:
        out = []
        for i in range(0, len(W), 2048):
            out.append(models[n].embed(torch.from_numpy(W[i:i + 2048]).to(DEVICE),
                                       torch.from_numpy(C[i:i + 2048]).to(DEVICE)).cpu().numpy())
        recs[n]['tk'] += [tk] * len(W); recs[n]['dt'] += D; recs[n]['emb'].append(np.concatenate(out))

for n in BACKBONES:
    with h5py.File(OUT_DIR / f'rl_embeddings_{n}_fixed.h5', 'w') as f:
        f.create_dataset('tickers', data=np.array(recs[n]['tk'], dtype='S10'), compression='gzip')
        f.create_dataset('dates', data=np.array(recs[n]['dt'], dtype='S10'), compression='gzip')
        f.create_dataset('embeddings', data=np.concatenate(recs[n]['emb']).astype(np.float32), compression='gzip')
    print(n, '저장 완료:', len(recs[n]['tk']), '개')

## 6. PPO 실행 — 백본 2종 × 시드 5개

환경·보상·하이퍼파라미터는 `rl_trading_colab.ipynb`의 `KospiPortfolioEnv`와 같다. 속도를 위해 날짜별 클러스터 평균 임베딩과 클러스터 수익률을 미리 계산해 두는 점만 다르다(계산 결과는 같다).

**이어서 실행**: 런타임이 끊기면 §0(설정·데이터 로드 셀) → 아래 셀을 다시 실행한다. 이미 `ppo_runs.csv`에 있는 (백본, 시드)는 건너뛴다.

In [ ]:
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

clusters = pd.read_csv(OUT_DIR / 'cluster_assignments_2021_2023.csv', dtype={'종목코드': str})
clusters['종목코드'] = clusters['종목코드'].str.zfill(6)
cl_of = dict(zip(clusters['종목코드'], clusters['cluster']))

ret_df = raw[['종목코드', 'Date', 'RetTarget_1d']].dropna()
ret_df = ret_df[ret_df['종목코드'].isin(cl_of)]
ret_df['c'] = ret_df['종목코드'].map(cl_of)
ret_df['d'] = ret_df['Date'].dt.strftime('%Y-%m-%d')
cl_ret = ret_df.groupby(['d', 'c'])['RetTarget_1d'].mean().unstack().reindex(columns=range(N_CLUSTERS)).fillna(0.0)

def load_cluster_embs(name):
    with h5py.File(OUT_DIR / f'rl_embeddings_{name}_fixed.h5', 'r') as f:
        tk = [x.decode() for x in f['tickers'][:]]; dt = [x.decode() for x in f['dates'][:]]
        E = f['embeddings'][:]
    df = pd.DataFrame({'tk': tk, 'd': dt}); df['c'] = df['tk'].map(cl_of)
    df = df.dropna(subset=['c']); idx = df.index.to_numpy()
    out = {}
    for (d, c), g in df.groupby(['d', 'c']).groups.items():
        out[(d, int(c))] = E[g].mean(0)
    dates = sorted({d for d, _ in out})
    M = np.zeros((len(dates), N_CLUSTERS, 64), np.float32)
    for i, d in enumerate(dates):
        for c in range(N_CLUSTERS):
            if (d, c) in out: M[i, c] = out[(d, c)]
    return dates, M

class KospiPortfolioEnv(gym.Env):
    # rl_trading_colab.ipynb와 동일한 동역학: 상태=클러스터 평균 임베딩+현재 비중, 행동=softmax 비중,
    # 보상=다음 날짜 클러스터 등가중 RetTarget_1d 수익률 − 거래비용
    def __init__(self, dates, M_by_date, cl_ret, tc=TC):
        super().__init__()
        self.dates, self.M, self.R, self.tc = dates, M_by_date, cl_ret, tc
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(N_CLUSTERS * 64 + N_CLUSTERS,), dtype=np.float32)
        self.action_space = spaces.Box(-3.0, 3.0, shape=(N_CLUSTERS,), dtype=np.float32)
    def _obs(self): return np.concatenate([self.M[self.dates[self.t]].ravel(), self.w]).astype(np.float32)
    def reset(self, seed=None, options=None):
        super().reset(seed=seed); self.t, self.w, self.v = 0, np.ones(N_CLUSTERS, np.float32) / N_CLUSTERS, 1.0
        return self._obs(), {}
    def step(self, a):
        a = np.exp(a - a.max()); a = a / a.sum()
        cost = self.tc * np.abs(a - self.w).sum()
        nd = self.dates[min(self.t + 1, len(self.dates) - 1)]
        rv = self.R.get(nd, np.zeros(N_CLUSTERS))
        r = float((a * rv).sum())
        # [2026-10-04 회계 수정] 다음 스텝 비용은 가격 변동 후 실제 보유 비중 대비로 계산
        g = 1.0 + r
        self.v *= (1 + r - cost); self.w = (a * (1 + rv) / g if g > 1e-12 else a).astype(np.float32); self.t += 1
        return self._obs(), float(r - cost), self.t >= len(self.dates) - 1, False, {'portfolio_value': self.v}

def perf(vals):
    r = np.diff(vals) / vals[:-1]
    return {'total_ret': vals[-1] / vals[0] - 1, 'sharpe': r.mean() / (r.std() + 1e-8) * np.sqrt(252),
            'mdd': float(((vals / np.maximum.accumulate(vals)) - 1).min())}

def backtest(env, policy=None):
    obs, _ = env.reset(); done, vals = False, [1.0]
    eq = np.ones(N_CLUSTERS, np.float32) / N_CLUSTERS
    while not done:
        a = policy.predict(obs, deterministic=True)[0] if policy else np.log(eq)
        obs, _, done, _, info = env.step(a); vals.append(info['portfolio_value'])
    return np.array(vals)
print('환경 정의 완료')

In [ ]:
%%time
runs_csv = OUT_DIR / 'ppo_runs.csv'
done_runs = pd.read_csv(runs_csv) if runs_csv.exists() else pd.DataFrame(columns=['backbone', 'seed'])

for name in BACKBONES:
    dates, M = load_cluster_embs(name)
    M_by_date = {d: M[i] for i, d in enumerate(dates)}
    R = {d: cl_ret.loc[d].to_numpy(np.float32) for d in cl_ret.index}
    tr_dates = [d for d in dates if RL_TRAIN_START <= d <= RL_TRAIN_END]
    te_dates = [d for d in dates if RL_TEST_START <= d <= RL_TEST_END]
    ew = perf(backtest(KospiPortfolioEnv(te_dates, M_by_date, R)))
    print(f'[{name}] 학습 {len(tr_dates)}일 / 테스트 {len(te_dates)}일 | 등가중 테스트 {ew}')
    for seed in SEEDS:
        if ((done_runs['backbone'] == name) & (done_runs['seed'] == seed)).any():
            print(f'  seed={seed} 이미 완료 — 건너뜀'); continue
        ppo = PPO('MlpPolicy', KospiPortfolioEnv(tr_dates, M_by_date, R), learning_rate=3e-4, n_steps=512,
                  batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
                  vf_coef=0.5, max_grad_norm=0.5, policy_kwargs=dict(net_arch=[256, 256]),
                  verbose=0, device='cpu', seed=seed)
        ppo.learn(total_timesteps=PPO_TIMESTEPS)
        res = perf(backtest(KospiPortfolioEnv(te_dates, M_by_date, R), ppo))
        row = {'backbone': name, 'seed': seed, **{f'ppo_{k}': v for k, v in res.items()},
               **{f'ew_{k}': v for k, v in ew.items()}}
        done_runs = pd.concat([done_runs, pd.DataFrame([row])], ignore_index=True)
        done_runs.to_csv(runs_csv, index=False)
        print(f'  seed={seed}: PPO {res["total_ret"]:+.2%} / 샤프 {res["sharpe"]:.3f} | 등가중 {ew["total_ret"]:+.2%} / {ew["sharpe"]:.3f}')

## 7. 요약과 판정

In [ ]:
runs = pd.read_csv(OUT_DIR / 'ppo_runs.csv')
runs['beats_ew'] = runs['ppo_sharpe'] > runs['ew_sharpe']
summary = runs.groupby('backbone').agg(
    n=('seed', 'count'), ret_mean=('ppo_total_ret', 'mean'), ret_std=('ppo_total_ret', 'std'),
    sharpe_mean=('ppo_sharpe', 'mean'), sharpe_std=('ppo_sharpe', 'std'),
    beats_ew=('beats_ew', 'sum'), ew_ret=('ew_total_ret', 'first'), ew_sharpe=('ew_sharpe', 'first'))
print(runs.round(4).to_string(index=False)); print(); print(summary.round(4).to_string())
summary.to_csv(OUT_DIR / 'ppo_summary.csv', encoding='utf-8-sig')
print('\n참고(원래 파이프라인, 누수·버그 포함, 시드 42 1회): iTransformer +137.22% / 2.473, MTGNN +0.28% / 0.100, 등가중 +34.33% / 1.043')

### 판정 기준 (논문 반영 방법)

- **두 백본 모두 대부분의 시드에서 등가중보다 샤프가 낮다** → 1.1절의 "KOSPI 단일 정책 실패" 유지. 수치를 이 노트북 값(평균±표준편차)으로 교체하고, 원래 +0.28%는 버그·누수가 있던 파이프라인 값이라 각주로 내린다.
- **결과가 백본에 따라 갈리거나 시드별로 엇갈린다** → 1.1절을 "KOSPI 결과는 백본·시드에 민감했다"로 바꾸고, 단일 정책 실패의 근거는 크립토(−75.68%)와 미국 confound 실험으로 옮긴다. 1.4절 기여 #1과 7장 종합표도 같은 방향으로 수정한다.
- **두 백본 모두 대부분 등가중을 이긴다** → "KOSPI 단일 정책 실패" 주장은 철회. 원래 실패는 누수·버그의 산물이었다고 명시한다(한계 절).

결과 CSV(`ppo_runs.csv`, `ppo_summary.csv`, `backbone_val_metrics.json`)를 Claude에게 넘기면 논문 수정안을 만든다.

## 8. 정적 틸트 점검 — PPO 우위가 '시장을 읽은 결과'인가, '고정 비중 쏠림'인가

§6에서 단일 PPO가 10/10 등가중을 이겼지만, 백본 검증 dir_acc(0.495/0.515)가 다수결 기준선(0.533)보다 낮아 임베딩에 예측력이 거의 없다.
그렇다면 PPO는 날짜별로 판단을 바꾼 게 아니라 2024–25년에 강했던 클러스터에 **고정 비중으로 기울였을** 가능성이 있다. 아래 셀은 이를 직접 대조한다.

| 비교 대상 | 의미 |
|---|---|
| PPO (재학습) | §6 결과 재현 확인용 — `ppo_runs.csv`와 같아야 함 |
| 정적: PPO 테스트 평균 비중 | PPO가 테스트에서 쓴 비중의 평균을 매일 고정 리밸런싱 (사후 귀속용, 미래정보 포함) |
| 정적: PPO 학습 평균 비중 | PPO가 학습 구간에서 쓴 비중의 평균을 고정 (미래정보 없음 — 공정한 대조) |
| 클러스터 c 단독 | 어느 클러스터가 수익을 냈는지 확인 |

**실행**: §0(설정·데이터 로드) → §6 첫 셀(환경 정의) → 아래 셀. PPO 모델은 백본별 시드 42를 다시 학습해 `OUT_DIR/ppo_{백본}_s{시드}.zip`으로 저장한다(백본당 약 10분, 다음부터는 로드만).

In [ ]:
%%time
STATIC_SEEDS = [42]   # 시간이 되면 SEEDS 전체로 늘려도 된다 (시드당 약 10분)

class ConstPolicy:
    """매 스텝 같은 목표 비중을 내는 정책 — env.step이 softmax를 취하므로 log(w)를 행동으로 넘긴다."""
    def __init__(self, w): self.a = np.log(np.clip(np.asarray(w, np.float64), 1e-8, None)).astype(np.float32)
    def predict(self, obs, deterministic=True): return self.a, None

def policy_weights(env, policy):
    """정책을 env에서 끝까지 돌리며 매 스텝 목표 비중(softmax 후)과 NAV를 기록."""
    obs, _ = env.reset(); done, vals, W = False, [1.0], []
    while not done:
        a = policy.predict(obs, deterministic=True)[0]
        e = np.exp(a - a.max()); W.append(e / e.sum())
        obs, _, done, _, info = env.step(a); vals.append(info['portfolio_value'])
    return np.array(W), np.array(vals)

runs_ref = pd.read_csv(OUT_DIR / 'ppo_runs.csv')
rows = []
for name in BACKBONES:
    dates, M = load_cluster_embs(name)
    M_by_date = {d: M[i] for i, d in enumerate(dates)}
    R = {d: cl_ret.loc[d].to_numpy(np.float32) for d in cl_ret.index}
    tr_dates = [d for d in dates if RL_TRAIN_START <= d <= RL_TRAIN_END]
    te_dates = [d for d in dates if RL_TEST_START <= d <= RL_TEST_END]
    te_env = lambda: KospiPortfolioEnv(te_dates, M_by_date, R)

    for seed in STATIC_SEEDS:
        path = OUT_DIR / f'ppo_{name}_s{seed}.zip'
        if path.exists():
            ppo = PPO.load(str(path), device='cpu'); print(f'[{name} s{seed}] 저장된 모델 로드')
        else:
            print(f'[{name} s{seed}] 재학습 중...')
            ppo = PPO('MlpPolicy', KospiPortfolioEnv(tr_dates, M_by_date, R), learning_rate=3e-4, n_steps=512,
                      batch_size=64, n_epochs=10, gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01,
                      vf_coef=0.5, max_grad_norm=0.5, policy_kwargs=dict(net_arch=[256, 256]),
                      verbose=0, device='cpu', seed=seed)
            ppo.learn(total_timesteps=PPO_TIMESTEPS); ppo.save(str(path))

        W_te, v_te = policy_weights(te_env(), ppo)
        W_tr, _ = policy_weights(KospiPortfolioEnv(tr_dates, M_by_date, R), ppo)
        w_te_mean, w_tr_mean = W_te.mean(0), W_tr.mean(0)

        ref = runs_ref[(runs_ref['backbone'] == name) & (runs_ref['seed'] == seed)]
        ppo_perf = perf(v_te)
        if len(ref):
            print(f'  재현 확인: 재학습 Sharpe {ppo_perf["sharpe"]:.3f} vs ppo_runs.csv {ref["ppo_sharpe"].iloc[0]:.3f}')

        cand = {'PPO': ppo_perf,
                '정적(테스트 평균 비중)': perf(backtest(te_env(), ConstPolicy(w_te_mean))),
                '정적(학습 평균 비중)': perf(backtest(te_env(), ConstPolicy(w_tr_mean))),
                '등가중': perf(backtest(te_env()))}
        for c in range(N_CLUSTERS):
            cand[f'클러스터 {c} 단독'] = perf(backtest(te_env(), ConstPolicy(np.eye(N_CLUSTERS)[c] + 1e-6)))
        for k, p in cand.items():
            rows.append({'backbone': name, 'seed': seed, 'strategy': k, **p})

        print(f'  테스트 평균 비중: {np.round(w_te_mean, 3)} | 학습 평균 비중: {np.round(w_tr_mean, 3)}')
        print(f'  비중의 시간 변동(클러스터별 표준편차 평균): {W_te.std(0).mean():.4f} | '
              f'일평균 비중 변화량: {np.abs(np.diff(W_te, axis=0)).sum(1).mean():.4f}')

static_df = pd.DataFrame(rows)
print(); print(static_df.round(4).to_string(index=False))
static_df.to_csv(OUT_DIR / 'ppo_static_tilt_check.csv', index=False, encoding='utf-8-sig')
print('\n저장 완료: ppo_static_tilt_check.csv')

### 판정 기준

- **PPO와 '정적(테스트 평균 비중)'의 Sharpe 차이가 0.05 이내이고, 비중의 시간 변동이 작다(표준편차 평균 < 0.05)** → PPO는 날짜별 판단 없이 고정 쏠림을 학습한 것. 논문에는 "KOSPI 단일 PPO는 등가중을 이겼지만, 그 우위는 고정 클러스터 비중으로 재현된다"로 쓴다.
- **'정적(학습 평균 비중)'도 등가중을 이긴다** → 학습 구간 정보만으로 고를 수 있는 쏠림이었다는 뜻(미래정보 없음). 이기지 못한다면 PPO의 쏠림 방향이 테스트 구간에 운 좋게 맞았던 것.
- **PPO가 두 정적 비중보다 뚜렷이 낫다** → 시간에 따른 판단(타이밍)이 실제로 기여한 것. 이 경우에만 "임베딩 기반 정책이 KOSPI에서 작동했다"고 쓸 수 있다.
- **재현 확인 Sharpe가 `ppo_runs.csv`와 다르면** → 환경 코드(10-04 회계 수정 등)가 §6 실행 때와 달라진 것이니, 이 셀의 PPO 행을 기준으로 해석한다.